In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
from scipy.optimize import curve_fit
from scipy.special import voigt_profile
from scipy.integrate import simpson
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

class PeakFittingApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("WAXS Multi-Peak Fitting with Individual Peak Shapes")
        self.showMaximized()
        
        # Data
        self.x_data = None
        self.y_data = None
        self.y_data_corrected = None
        self.initial_peak_centers = []
        self.fit_results = None
        self.peak_params = []
        
        # Fitting boundaries
        self.fit_bounds = [None, None]
        self.selecting_bounds = False
        self.bounds_rect = None
        
        # Background parameters
        self.background_coeff = None
        self.background_subtracted = False
        
        # Lorentz correction
        self.lorentz_corrected = False
        
        # Peak settings - INDIVIDUAL FOR EACH PEAK
        self.peak_shapes = []  # Individual shape for each peak: 'Gaussian', 'Lorentzian', 'Voigt'
        self.default_shape = 'Gaussian'
        self.n_peaks = 5
        self.peak_tolerances = []
        self.default_tolerance = 0.1
        
        # Peak types
        self.peak_types = []
        
        # Colors
        self.colors = ['red', 'blue', 'green', 'orange', 'purple', 
                      'brown', 'pink', 'gray', 'olive', 'cyan']
        
        # Interaction mode
        self.interaction_mode = 'navigation'
        
        self.init_ui()
        
    def init_ui(self):
        # Central widget
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QHBoxLayout(central_widget)
        main_layout.setContentsMargins(5, 5, 5, 5)
        main_layout.setSpacing(5)
        
        # Left panel - plot
        left_panel = QWidget()
        left_layout = QVBoxLayout(left_panel)
        left_layout.setContentsMargins(0, 0, 0, 0)
        
        # Create figure
        self.figure = Figure(figsize=(8, 8))
        self.canvas = FigureCanvas(self.figure)
        self.ax = self.figure.add_subplot(111)
        
        # Add toolbar
        self.toolbar = NavigationToolbar(self.canvas, self)
        
        left_layout.addWidget(self.toolbar)
        left_layout.addWidget(self.canvas)
        
        # Connect mouse events
        self.canvas.mpl_connect('button_press_event', self.on_click)
        self.canvas.mpl_connect('button_release_event', self.on_release)
        self.canvas.mpl_connect('motion_notify_event', self.on_motion)
        
        # Right panel - controls
        right_panel = QWidget()
        right_panel.setFixedWidth(650)  # Slightly wider for better visibility
        right_layout = QVBoxLayout(right_panel)  # Changed to VBox for single column layout
        right_layout.setContentsMargins(5, 5, 5, 5)
        right_layout.setSpacing(8)
        
        # Title
        title = QLabel("FITTING CONTROLS")
        title.setStyleSheet("font-size: 14px; font-weight: bold; padding: 5px; background-color: #e0e0e0;")
        title.setAlignment(Qt.AlignCenter)
        right_layout.addWidget(title)
        
        # TOP ROW - Two columns for compact controls
        top_row = QWidget()
        top_layout = QHBoxLayout(top_row)
        top_layout.setContentsMargins(0, 0, 0, 0)
        top_layout.setSpacing(5)
        
        # LEFT TOP COLUMN
        left_top = QWidget()
        left_top_layout = QVBoxLayout(left_top)
        left_top_layout.setContentsMargins(0, 0, 0, 0)
        left_top_layout.setSpacing(8)
        
        # Data loading group
        load_group = QGroupBox("Data Loading")
        load_layout = QVBoxLayout()
        load_layout.setSpacing(3)
        
        self.load_btn = QPushButton("Load Data (.txt/.csv)")
        self.load_btn.clicked.connect(self.load_data)
        load_layout.addWidget(self.load_btn)
        
        self.data_info = QLabel("No data loaded")
        self.data_info.setWordWrap(True)
        self.data_info.setStyleSheet("background-color: #f0f0f0; padding: 5px;")
        load_layout.addWidget(self.data_info)
        
        load_group.setLayout(load_layout)
        left_top_layout.addWidget(load_group)
        
        # Interaction mode group
        mode_group = QGroupBox("Mode")
        mode_layout = QVBoxLayout()
        mode_layout.setSpacing(3)
        
        self.nav_mode_btn = QPushButton("🔍 Navigate")
        self.nav_mode_btn.setCheckable(True)
        self.nav_mode_btn.setChecked(True)
        self.nav_mode_btn.clicked.connect(lambda: self.set_interaction_mode('navigation'))
        
        self.peak_mode_btn = QPushButton("⛰️ Select Peaks")
        self.peak_mode_btn.setCheckable(True)
        self.peak_mode_btn.clicked.connect(lambda: self.set_interaction_mode('peak_selection'))
        
        self.boundary_mode_btn = QPushButton("📏 Set Boundaries")
        self.boundary_mode_btn.setCheckable(True)
        self.boundary_mode_btn.clicked.connect(lambda: self.set_interaction_mode('boundary_selection'))
        
        mode_layout.addWidget(self.nav_mode_btn)
        mode_layout.addWidget(self.peak_mode_btn)
        mode_layout.addWidget(self.boundary_mode_btn)
        
        self.mode_label = QLabel("Navigate")
        self.mode_label.setStyleSheet("background-color: #e8f4f8; padding: 3px;")
        self.mode_label.setAlignment(Qt.AlignCenter)
        mode_layout.addWidget(self.mode_label)
        
        mode_group.setLayout(mode_layout)
        left_top_layout.addWidget(mode_group)
        
        # Fitting boundaries group
        bounds_group = QGroupBox("Boundaries")
        bounds_layout = QVBoxLayout()
        bounds_layout.setSpacing(3)
        
        bounds_info_layout = QHBoxLayout()
        bounds_info_layout.addWidget(QLabel("Left:"))
        self.left_bound_label = QLabel("--")
        self.left_bound_label.setStyleSheet("background-color: #f0f0f0; padding: 3px;")
        bounds_info_layout.addWidget(self.left_bound_label)
        bounds_info_layout.addWidget(QLabel("Right:"))
        self.right_bound_label = QLabel("--")
        self.right_bound_label.setStyleSheet("background-color: #f0f0f0; padding: 3px;")
        bounds_info_layout.addWidget(self.right_bound_label)
        bounds_layout.addLayout(bounds_info_layout)
        
        self.clear_bounds_btn = QPushButton("Clear Boundaries")
        self.clear_bounds_btn.clicked.connect(self.clear_bounds)
        bounds_layout.addWidget(self.clear_bounds_btn)
        
        bounds_group.setLayout(bounds_layout)
        left_top_layout.addWidget(bounds_group)
        
        # Add stretch
        left_top_layout.addStretch()
        
        # RIGHT TOP COLUMN
        right_top = QWidget()
        right_top_layout = QVBoxLayout(right_top)
        right_top_layout.setContentsMargins(0, 0, 0, 0)
        right_top_layout.setSpacing(8)
        
        # Data preprocessing group
        preprocess_group = QGroupBox("Preprocessing")
        preprocess_layout = QVBoxLayout()
        preprocess_layout.setSpacing(3)
        
        self.bg_subtract_btn = QPushButton("1. Subtract Linear Background")
        self.bg_subtract_btn.clicked.connect(self.subtract_background)
        self.bg_subtract_btn.setEnabled(False)
        
        self.lorentz_btn = QPushButton("2. Apply Lorentz Correction (×q²)")
        self.lorentz_btn.clicked.connect(self.apply_lorentz_correction)
        self.lorentz_btn.setEnabled(False)
        
        self.reset_btn = QPushButton("Reset to Original Data")
        self.reset_btn.clicked.connect(self.reset_to_original)
        self.reset_btn.setEnabled(False)
        
        preprocess_layout.addWidget(self.bg_subtract_btn)
        preprocess_layout.addWidget(self.lorentz_btn)
        preprocess_layout.addWidget(self.reset_btn)
        
        self.data_status = QLabel("Original data")
        self.data_status.setStyleSheet("background-color: #e8f4f8; padding: 3px;")
        self.data_status.setAlignment(Qt.AlignCenter)
        preprocess_layout.addWidget(self.data_status)
        
        preprocess_group.setLayout(preprocess_layout)
        right_top_layout.addWidget(preprocess_group)
        
        # Peak settings group
        peaks_group = QGroupBox("Default Settings")
        peaks_layout = QVBoxLayout()
        peaks_layout.setSpacing(3)
        
        # Number of peaks
        n_peaks_layout = QHBoxLayout()
        n_peaks_layout.addWidget(QLabel("Number of peaks:"))
        self.n_peaks_spin = QSpinBox()
        self.n_peaks_spin.setRange(1, 10)
        self.n_peaks_spin.setValue(5)
        self.n_peaks_spin.valueChanged.connect(self.update_n_peaks)
        n_peaks_layout.addWidget(self.n_peaks_spin)
        peaks_layout.addLayout(n_peaks_layout)
        
        # Default peak shape
        shape_layout = QHBoxLayout()
        shape_layout.addWidget(QLabel("Default shape:"))
        self.default_shape_combo = QComboBox()
        self.default_shape_combo.addItems(['Gaussian', 'Lorentzian', 'Voigt'])
        self.default_shape_combo.currentTextChanged.connect(self.update_default_shape)
        shape_layout.addWidget(self.default_shape_combo)
        peaks_layout.addLayout(shape_layout)
        
        # Default tolerance
        tol_layout = QHBoxLayout()
        tol_layout.addWidget(QLabel("Default tolerance:"))
        self.default_tolerance_spin = QDoubleSpinBox()
        self.default_tolerance_spin.setRange(0.1, 30.0)
        self.default_tolerance_spin.setValue(10.0)
        self.default_tolerance_spin.setSuffix(" %")
        self.default_tolerance_spin.valueChanged.connect(self.update_default_tolerance)
        tol_layout.addWidget(self.default_tolerance_spin)
        peaks_layout.addLayout(tol_layout)
        
        peaks_group.setLayout(peaks_layout)
        right_top_layout.addWidget(peaks_group)
        
        # Crystallinity group (compact)
        crystallinity_group = QGroupBox("Crystallinity")
        crystallinity_layout = QVBoxLayout()
        crystallinity_layout.setSpacing(3)
        
        self.calc_crystallinity_btn = QPushButton("Calculate")
        self.calc_crystallinity_btn.clicked.connect(self.calculate_crystallinity)
        self.calc_crystallinity_btn.setEnabled(False)
        
        self.crystallinity_result = QLabel("-- %")
        self.crystallinity_result.setStyleSheet("background-color: #e8f4f8; padding: 8px; font-size: 14px; font-weight: bold;")
        self.crystallinity_result.setAlignment(Qt.AlignCenter)
        
        crystallinity_layout.addWidget(self.calc_crystallinity_btn)
        crystallinity_layout.addWidget(self.crystallinity_result)
        
        crystallinity_group.setLayout(crystallinity_layout)
        right_top_layout.addWidget(crystallinity_group)
        
        # Add both top columns
        top_layout.addWidget(left_top, 1)
        top_layout.addWidget(right_top, 1)
        
        right_layout.addWidget(top_row)
        
        # SELECTED PEAKS TABLE (full width)
        peaks_list_group = QGroupBox("Selected Peaks - Individual Settings")
        peaks_list_layout = QVBoxLayout()
        peaks_list_layout.setSpacing(3)
        
        # Enhanced table with 5 columns: Position, Type, Shape, Tolerance(%), Action
        self.peaks_table = QTableWidget()
        self.peaks_table.setColumnCount(5)
        self.peaks_table.setHorizontalHeaderLabels(["Position", "Type", "Shape", "Tol(%)", ""])
        self.peaks_table.setMaximumHeight(180)
        self.peaks_table.setMinimumHeight(150)
        self.peaks_table.itemChanged.connect(self.on_tolerance_changed)
        peaks_list_layout.addWidget(self.peaks_table)
        
        # Buttons row
        btn_row = QHBoxLayout()
        self.clear_peaks_btn = QPushButton("Clear All Peaks")
        self.clear_peaks_btn.clicked.connect(self.clear_peaks)
        self.remove_last_btn = QPushButton("Remove Last Peak")
        self.remove_last_btn.clicked.connect(self.remove_last_peak)
        btn_row.addWidget(self.clear_peaks_btn)
        btn_row.addWidget(self.remove_last_btn)
        peaks_list_layout.addLayout(btn_row)
        
        # Quick presets row
        preset_row = QHBoxLayout()
        preset_row.addWidget(QLabel("Quick presets for selected:"))
        
        self.set_crystalline_btn = QPushButton("Crystalline")
        self.set_crystalline_btn.clicked.connect(lambda: self.set_peak_type('crystalline'))
        
        self.set_amorphous_btn = QPushButton("Amorphous")
        self.set_amorphous_btn.clicked.connect(lambda: self.set_peak_type('amorphous'))
        
        self.set_gaussian_btn = QPushButton("Gaussian")
        self.set_gaussian_btn.clicked.connect(lambda: self.set_peak_shape('Gaussian'))
        
        self.set_lorentzian_btn = QPushButton("Lorentzian")
        self.set_lorentzian_btn.clicked.connect(lambda: self.set_peak_shape('Lorentzian'))
        
        self.set_voigt_btn = QPushButton("Voigt")
        self.set_voigt_btn.clicked.connect(lambda: self.set_peak_shape('Voigt'))
        
        preset_row.addWidget(self.set_crystalline_btn)
        preset_row.addWidget(self.set_amorphous_btn)
        preset_row.addWidget(self.set_gaussian_btn)
        preset_row.addWidget(self.set_lorentzian_btn)
        preset_row.addWidget(self.set_voigt_btn)
        peaks_list_layout.addLayout(preset_row)
        
        # Tolerance presets
        tol_preset_row = QHBoxLayout()
        tol_preset_row.addWidget(QLabel("Tolerance presets:"))
        self.set_tol_5_btn = QPushButton("5%")
        self.set_tol_5_btn.clicked.connect(lambda: self.set_tolerance_for_selected(5.0))
        self.set_tol_10_btn = QPushButton("10%")
        self.set_tol_10_btn.clicked.connect(lambda: self.set_tolerance_for_selected(10.0))
        self.set_tol_15_btn = QPushButton("15%")
        self.set_tol_15_btn.clicked.connect(lambda: self.set_tolerance_for_selected(15.0))
        self.set_tol_20_btn = QPushButton("20%")
        self.set_tol_20_btn.clicked.connect(lambda: self.set_tolerance_for_selected(20.0))
        
        tol_preset_row.addWidget(self.set_tol_5_btn)
        tol_preset_row.addWidget(self.set_tol_10_btn)
        tol_preset_row.addWidget(self.set_tol_15_btn)
        tol_preset_row.addWidget(self.set_tol_20_btn)
        peaks_list_layout.addLayout(tol_preset_row)
        
        peaks_list_group.setLayout(peaks_list_layout)
        right_layout.addWidget(peaks_list_group)
        
        # FITTING BUTTON (full width)
        self.fit_btn = QPushButton("PERFORM FITTING")
        self.fit_btn.setStyleSheet("background-color: #4CAF50; color: white; font-weight: bold; padding: 10px; font-size: 14px;")
        self.fit_btn.clicked.connect(self.perform_fitting)
        self.fit_btn.setEnabled(False)
        right_layout.addWidget(self.fit_btn)
        
        # RESULTS - Full width and larger
        results_group = QGroupBox("Detailed Results")
        results_layout = QVBoxLayout()
        results_layout.setSpacing(3)
        
        self.results_text = QTextEdit()
        self.results_text.setReadOnly(True)
        self.results_text.setMaximumHeight(300)
        self.results_text.setMinimumHeight(250)
        self.results_text.setFont(QFont("Courier", 10))
        results_layout.addWidget(self.results_text)
        
        # Save button
        self.save_btn = QPushButton("Save Results to File")
        self.save_btn.clicked.connect(self.save_results)
        self.save_btn.setEnabled(False)
        results_layout.addWidget(self.save_btn)
        
        results_group.setLayout(results_layout)
        right_layout.addWidget(results_group)
        
        # Add panels to main layout
        main_layout.addWidget(left_panel, 1)
        main_layout.addWidget(right_panel)
        
    def set_interaction_mode(self, mode):
        self.interaction_mode = mode
        
        self.nav_mode_btn.setChecked(mode == 'navigation')
        self.peak_mode_btn.setChecked(mode == 'peak_selection')
        self.boundary_mode_btn.setChecked(mode == 'boundary_selection')
        
        if mode == 'navigation':
            self.mode_label.setText("Navigation mode - Use toolbar to zoom/pan")
            self.mode_label.setStyleSheet("background-color: #e8f4f8; padding: 3px;")
        elif mode == 'peak_selection':
            self.mode_label.setText("Peak selection mode - Click to select peaks")
            self.mode_label.setStyleSheet("background-color: #fff3cd; padding: 3px;")
        elif mode == 'boundary_selection':
            self.mode_label.setText("Boundary mode - Click and drag to set limits")
            self.mode_label.setStyleSheet("background-color: #f8d7da; padding: 3px;")
            
        if mode != 'boundary_selection':
            self.selecting_bounds = False
            self.bounds_rect = None
            
    def update_default_shape(self, shape):
        self.default_shape = shape
        
    def update_default_tolerance(self, value):
        self.default_tolerance = value / 100.0
        
    def set_peak_shape(self, shape):
        """Set shape for selected peaks"""
        selected_rows = set()
        for item in self.peaks_table.selectedItems():
            selected_rows.add(item.row())
            
        for row in selected_rows:
            if row < len(self.peak_shapes):
                self.peak_shapes[row] = shape
                
        self.update_peaks_table()
        self.plot_data()
        
    def on_tolerance_changed(self, item):
        if item.column() == 3:  # Tolerance column
            row = item.row()
            try:
                new_tol = float(item.text()) / 100.0
                if row < len(self.peak_tolerances):
                    self.peak_tolerances[row] = new_tol
                    self.plot_data()
            except ValueError:
                pass
                
    def set_tolerance_for_selected(self, tolerance_percent):
        selected_rows = set()
        for item in self.peaks_table.selectedItems():
            selected_rows.add(item.row())
            
        for row in selected_rows:
            if row < len(self.peak_tolerances):
                self.peak_tolerances[row] = tolerance_percent / 100.0
                
        self.update_peaks_table()
        self.plot_data()
        
    def on_click(self, event):
        if event.inaxes != self.ax or self.x_data is None:
            return
            
        if self.interaction_mode == 'boundary_selection' and event.button == 1:
            self.fit_bounds[0] = event.xdata
            y_min = min(self.get_current_y_data())
            y_max = max(self.get_current_y_data())
            self.bounds_rect = Rectangle((event.xdata, y_min), 
                                        0, y_max - y_min,
                                        alpha=0.3, color='yellow')
            self.ax.add_patch(self.bounds_rect)
            self.canvas.draw()
            
        elif self.interaction_mode == 'peak_selection' and event.button == 1:
            if len(self.initial_peak_centers) < self.n_peaks:
                x_click = event.xdata
                self.initial_peak_centers.append(x_click)
                self.peak_types.append('crystalline')
                self.peak_shapes.append(self.default_shape)
                self.peak_tolerances.append(self.default_tolerance)
                self.update_peaks_table()
                self.plot_data()
                
    def on_motion(self, event):
        if (self.interaction_mode == 'boundary_selection' and 
            self.bounds_rect is not None and event.inaxes == self.ax):
            width = event.xdata - self.fit_bounds[0]
            self.bounds_rect.set_width(width)
            self.canvas.draw()
            
    def on_release(self, event):
        if (self.interaction_mode == 'boundary_selection' and 
            self.bounds_rect is not None and event.inaxes == self.ax):
            self.fit_bounds[1] = event.xdata
            if self.fit_bounds[0] > self.fit_bounds[1]:
                self.fit_bounds[0], self.fit_bounds[1] = self.fit_bounds[1], self.fit_bounds[0]
                
            self.left_bound_label.setText(f"{self.fit_bounds[0]:.4f}")
            self.right_bound_label.setText(f"{self.fit_bounds[1]:.4f}")
            
            self.bg_subtract_btn.setEnabled(True)
            self.plot_data()
            self.set_interaction_mode('navigation')
            
    def clear_bounds(self):
        self.fit_bounds = [None, None]
        self.left_bound_label.setText("--")
        self.right_bound_label.setText("--")
        self.bg_subtract_btn.setEnabled(False)
        self.lorentz_btn.setEnabled(False)
        self.plot_data()
        
    def get_current_y_data(self):
        if self.lorentz_corrected and self.y_data_corrected is not None:
            return self.y_data_corrected
        elif self.background_subtracted and self.y_data_corrected is not None:
            return self.y_data_corrected
        else:
            return self.y_data
            
    def subtract_background(self):
        if self.x_data is None or self.fit_bounds[0] is None:
            return
            
        left_idx = np.argmin(np.abs(self.x_data - self.fit_bounds[0]))
        right_idx = np.argmin(np.abs(self.x_data - self.fit_bounds[1]))
        
        x1, y1 = self.x_data[left_idx], self.y_data[left_idx]
        x2, y2 = self.x_data[right_idx], self.y_data[right_idx]
        
        slope = (y2 - y1) / (x2 - x1)
        intercept = y1 - slope * x1
        
        self.background_coeff = [slope, intercept]
        
        background = intercept + slope * self.x_data
        self.y_data_corrected = self.y_data - background
        
        self.background_subtracted = True
        self.lorentz_corrected = False
        self.lorentz_btn.setEnabled(True)
        self.reset_btn.setEnabled(True)
        self.data_status.setText("Background subtracted")
        
        self.clear_peaks()
        self.plot_data()
        
    def apply_lorentz_correction(self):
        if self.x_data is None:
            return
            
        if not self.background_subtracted or self.y_data_corrected is None:
            y_to_correct = self.y_data
        else:
            y_to_correct = self.y_data_corrected
            
        x_safe = np.where(self.x_data > 0, self.x_data, 1e-6)
        self.y_data_corrected = y_to_correct * x_safe**2
        
        self.lorentz_corrected = True
        self.data_status.setText("Lorentz corrected")
        
        self.clear_peaks()
        self.plot_data()
        
    def reset_to_original(self):
        self.y_data_corrected = None
        self.background_subtracted = False
        self.lorentz_corrected = False
        self.background_coeff = None
        self.data_status.setText("Original data")
        self.lorentz_btn.setEnabled(False)
        self.reset_btn.setEnabled(False)
        
        self.clear_peaks()
        self.plot_data()
        
    def update_n_peaks(self, value):
        self.n_peaks = value
        
    def load_data(self):
        filename, _ = QFileDialog.getOpenFileName(
            self, "Select data file", "", "Text files (*.txt *.csv *.dat)")
        
        if filename:
            try:
                data = np.loadtxt(filename)
                if data.shape[1] >= 2:
                    self.x_data = data[:, 0]
                    self.y_data = data[:, 1]
                    self.y_data_corrected = None
                    self.background_subtracted = False
                    self.lorentz_corrected = False
                    self.background_coeff = None
                    
                    self.data_info.setText(f"File: {filename.split('/')[-1]}\n"
                                          f"Points: {len(self.x_data)}\n"
                                          f"Range: [{self.x_data[0]:.3f}, {self.x_data[-1]:.3f}]")
                    
                    self.data_status.setText("Original data")
                    self.fit_btn.setEnabled(True)
                    self.reset_btn.setEnabled(False)
                    self.lorentz_btn.setEnabled(False)
                    self.bg_subtract_btn.setEnabled(False)
                    self.plot_data()
                else:
                    QMessageBox.warning(self, "Error", "File must contain 2 columns (x and y)")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Could not load file:\n{str(e)}")
                
    def plot_data(self):
        """Plot data while preserving current view limits"""
        
        # Save current view limits
        if self.x_data is not None and hasattr(self, 'ax'):
            try:
                xlim = self.ax.get_xlim()
                ylim = self.ax.get_ylim()
                has_limits = True
            except:
                has_limits = False
        else:
            has_limits = False
        
        self.ax.clear()
        
        if self.x_data is None:
            return
        
        y_plot_data = self.get_current_y_data()
        
        # Plot data
        if self.fit_bounds[0] is not None and self.fit_bounds[1] is not None:
            mask = (self.x_data >= self.fit_bounds[0]) & (self.x_data <= self.fit_bounds[1])
            x_plot = self.x_data[mask]
            y_plot = y_plot_data[mask]
            
            self.ax.plot(self.x_data, y_plot_data, 'k-', linewidth=1, alpha=0.3, label='Full data')
            self.ax.plot(x_plot, y_plot, 'k-', linewidth=2, label='Fit region')
            
            if self.background_coeff is not None and not self.lorentz_corrected:
                bg_line = self.background_coeff[1] + self.background_coeff[0] * self.x_data
                self.ax.plot(self.x_data, bg_line, 'g--', linewidth=1, alpha=0.5, label='Background')
            
            self.ax.axvline(x=self.fit_bounds[0], color='red', linestyle='--', alpha=0.7)
            self.ax.axvline(x=self.fit_bounds[1], color='red', linestyle='--', alpha=0.7)
        else:
            x_plot = self.x_data
            y_plot = y_plot_data
            self.ax.plot(x_plot, y_plot, 'k-', linewidth=1.5, label='Data')
        
        # Mark initial peak positions with corridors
        x_range = x_plot[-1] - x_plot[0]
        for i, center in enumerate(self.initial_peak_centers):
            if center >= x_plot[0] and center <= x_plot[-1]:
                color = self.colors[i % len(self.colors)]
                
                # Show corridor
                if self.fit_results is None and i < len(self.peak_tolerances):
                    delta = x_range * self.peak_tolerances[i]
                    self.ax.axvspan(center - delta, center + delta, 
                                   alpha=0.15, color=color)
                    
                    # Enhanced label with shape info
                    shape_code = {'Gaussian': 'G', 'Lorentzian': 'L', 'Voigt': 'V'}[self.peak_shapes[i]]
                    label = f'P{i+1}({shape_code},{"C" if self.peak_types[i] == "crystalline" else "A"})'
                    y_pos = max(y_plot) * (0.95 - i * 0.04)
                    self.ax.text(center, y_pos, label, color=color, 
                               fontweight='bold', ha='center', fontsize=8)
        
        # If fitting results exist, show fits
        if self.fit_results is not None:
            self.plot_fit_results(x_plot)
        
        # Labels
        if self.lorentz_corrected:
            self.ax.set_ylabel('Intensity × q² (Lorentz corrected)')
        elif self.background_subtracted:
            self.ax.set_ylabel('Intensity (background subtracted)')
        else:
            self.ax.set_ylabel('Intensity')
        
        self.ax.set_xlabel('q (scattering vector)')
        self.ax.legend(loc='upper right', fontsize=8)
        self.ax.grid(True, alpha=0.3)
        
        # Restore view
        if has_limits:
            try:
                if (xlim[0] >= min(self.x_data) and xlim[1] <= max(self.x_data)):
                    self.ax.set_xlim(xlim)
                    self.ax.set_ylim(ylim)
            except:
                pass
        
        self.canvas.draw()
        
    def update_peaks_table(self):
        """Update peaks table with individual settings"""
        self.peaks_table.blockSignals(True)
        self.peaks_table.setRowCount(len(self.initial_peak_centers))
        
        for i, (center, ptype, shape, tol) in enumerate(zip(self.initial_peak_centers, 
                                                            self.peak_types, 
                                                            self.peak_shapes, 
                                                            self.peak_tolerances)):
            # Position
            pos_item = QTableWidgetItem(f"{center:.4f}")
            pos_item.setFlags(pos_item.flags() & ~Qt.ItemIsEditable)
            self.peaks_table.setItem(i, 0, pos_item)
            
            # Type
            type_item = QTableWidgetItem(ptype.capitalize())
            type_item.setFlags(type_item.flags() & ~Qt.ItemIsEditable)
            if ptype == 'crystalline':
                type_item.setForeground(QColor('blue'))
                type_item.setBackground(QColor(230, 240, 255))
            else:
                type_item.setForeground(QColor('red'))
                type_item.setBackground(QColor(255, 230, 230))
            self.peaks_table.setItem(i, 1, type_item)
            
            # Shape
            shape_item = QTableWidgetItem(shape)
            shape_item.setFlags(shape_item.flags() & ~Qt.ItemIsEditable)
            shape_item.setForeground(QColor('darkgreen'))
            shape_item.setBackground(QColor(230, 255, 230))
            self.peaks_table.setItem(i, 2, shape_item)
            
            # Tolerance
            tol_item = QTableWidgetItem(f"{tol*100:.1f}")
            tol_item.setForeground(QColor('darkorange'))
            tol_item.setBackground(QColor(255, 240, 200))
            self.peaks_table.setItem(i, 3, tol_item)
            
            # Action button
            btn = QPushButton("🗑️")
            btn.clicked.connect(lambda checked, row=i: self.remove_specific_peak(row))
            self.peaks_table.setCellWidget(i, 4, btn)
            
        self.peaks_table.blockSignals(False)
        
    def remove_specific_peak(self, row):
        """Remove a specific peak by row"""
        if 0 <= row < len(self.initial_peak_centers):
            self.initial_peak_centers.pop(row)
            self.peak_types.pop(row)
            self.peak_shapes.pop(row)
            self.peak_tolerances.pop(row)
            self.fit_results = None
            self.peak_params = []
            self.results_text.clear()
            self.calc_crystallinity_btn.setEnabled(False)
            self.crystallinity_result.setText("-- %")
            self.save_btn.setEnabled(False)
            self.update_peaks_table()
            self.plot_data()
            
    def set_peak_type(self, peak_type):
        selected_rows = set()
        for item in self.peaks_table.selectedItems():
            selected_rows.add(item.row())
            
        for row in selected_rows:
            if row < len(self.peak_types):
                self.peak_types[row] = peak_type
                
        self.update_peaks_table()
        self.plot_data()
        
    def clear_peaks(self):
        self.initial_peak_centers = []
        self.peak_types = []
        self.peak_shapes = []
        self.peak_tolerances = []
        self.fit_results = None
        self.peak_params = []
        self.results_text.clear()
        self.calc_crystallinity_btn.setEnabled(False)
        self.crystallinity_result.setText("-- %")
        self.save_btn.setEnabled(False)
        self.update_peaks_table()
        self.plot_data()
        
    def remove_last_peak(self):
        if self.initial_peak_centers:
            self.initial_peak_centers.pop()
            self.peak_types.pop()
            self.peak_shapes.pop()
            self.peak_tolerances.pop()
            self.fit_results = None
            self.peak_params = []
            self.results_text.clear()
            self.calc_crystallinity_btn.setEnabled(False)
            self.crystallinity_result.setText("-- %")
            self.save_btn.setEnabled(False)
            self.update_peaks_table()
            self.plot_data()
            
    def gaussian(self, x, amp, cen, sigma):
        return amp * np.exp(-(x - cen)**2 / (2 * sigma**2))
    
    def lorentzian(self, x, amp, cen, gamma):
        return amp * (gamma**2 / ((x - cen)**2 + gamma**2))
    
    def voigt(self, x, amp, cen, sigma, gamma):
        return amp * voigt_profile(x - cen, sigma, gamma)
    
    def multi_peak_model(self, x, *params):
        """Model with INDIVIDUAL SHAPES for each peak"""
        n_peaks = len(self.initial_peak_centers)
        
        # Linear background
        offset = params[0]
        slope = params[1]
        background = offset + slope * x
        
        # Sum of peaks with individual shapes
        peaks_sum = np.zeros_like(x)
        idx = 2
        for i in range(n_peaks):
            center = params[idx]
            amp = params[idx + 1]
            sigma = params[idx + 2]
            
            shape = self.peak_shapes[i]
            
            if shape == 'Gaussian':
                peaks_sum += self.gaussian(x, amp, center, sigma)
            elif shape == 'Lorentzian':
                peaks_sum += self.lorentzian(x, amp, center, sigma)
            else:  # Voigt
                gamma = params[idx + 3] if len(params) > idx + 3 else sigma / 2
                peaks_sum += self.voigt(x, amp, center, sigma, gamma)
                if shape == 'Voigt':
                    idx += 1
            
            idx += 3
            
        return background + peaks_sum
    
    def perform_fitting(self):
        if self.x_data is None or len(self.initial_peak_centers) == 0:
            QMessageBox.warning(self, "Warning", "Please select peaks first!")
            return
            
        try:
            y_current = self.get_current_y_data()
            
            if self.fit_bounds[0] is not None and self.fit_bounds[1] is not None:
                mask = (self.x_data >= self.fit_bounds[0]) & (self.x_data <= self.fit_bounds[1])
                x_fit = self.x_data[mask]
                y_fit = y_current[mask]
            else:
                x_fit = self.x_data
                y_fit = y_current
                
            n_peaks = len(self.initial_peak_centers)
            x_range = x_fit[-1] - x_fit[0]
            
            # Initial guesses
            p0 = []
            
            # Background
            margin = int(len(x_fit) * 0.1)
            if margin < 10:
                margin = min(10, len(x_fit) // 3)
                
            x_bg = np.concatenate([x_fit[:margin], x_fit[-margin:]])
            y_bg = np.concatenate([y_fit[:margin], y_fit[-margin:]])
            bg_coeff = np.polyfit(x_bg, y_bg, 1)
            p0.extend([bg_coeff[1], bg_coeff[0]])
            
            bounds_lower = [-np.inf, -np.inf]
            bounds_upper = [np.inf, np.inf]
            
            base_sigma = 1
            
            for i, init_center in enumerate(self.initial_peak_centers):
                delta = x_range * self.peak_tolerances[i]
                
                # Center
                p0.append(init_center)
                bounds_lower.append(init_center - delta)
                bounds_upper.append(init_center + delta)
                
                # Amplitude
                idx = np.argmin(np.abs(x_fit - init_center))
                amp0 = y_fit[idx] - np.polyval(bg_coeff, init_center)
                if amp0 < 0:
                    amp0 = y_fit[idx] * 0.5
                p0.append(max(amp0, 0.1))
                bounds_lower.append(0)
                bounds_upper.append(np.inf)
                
                # Sigma based on type
                if self.peak_types[i] == 'crystalline':
                    sigma0 = base_sigma * 0.5
                    sigma_max = x_range / 8
                else:
                    sigma0 = base_sigma * 4.0
                    sigma_max = x_range / 3
                    
                p0.append(sigma0)
                bounds_lower.append(1e-3)
                bounds_upper.append(sigma_max)
                
                # Gamma for Voigt
                if self.peak_shapes[i] == 'Voigt':
                    if self.peak_types[i] == 'crystalline':
                        gamma0 = sigma0 * 0.5
                    else:
                        gamma0 = sigma0 * 0.7
                    p0.append(gamma0)
                    bounds_lower.append(1e-4)
                    bounds_upper.append(x_range / 2)
                    
            popt, pcov = curve_fit(self.multi_peak_model, x_fit, y_fit, 
                                   p0=p0, bounds=(bounds_lower, bounds_upper), 
                                   maxfev=10000)
            
            self.fit_results = popt
            
            # Save results
            self.peak_params = []
            idx = 2
            for i, init_center in enumerate(self.initial_peak_centers):
                center = popt[idx]
                amp = popt[idx + 1]
                sigma = popt[idx + 2]
                gamma = popt[idx + 3] if self.peak_shapes[i] == 'Voigt' else sigma
                
                # FWHM calculation
                if self.peak_shapes[i] == 'Gaussian':
                    fwhm = 2.355 * sigma
                elif self.peak_shapes[i] == 'Lorentzian':
                    fwhm = 2 * sigma
                else:  # Voigt
                    fwhm = 0.5346 * (2.355 * sigma) + 0.2166 * (2 * gamma) + \
                           np.sqrt(0.2166**2 * (2*gamma)**2 + 0.5346**2 * (2.355*sigma)**2)
                
                self.peak_params.append({
                    'initial_center': init_center,
                    'center': center,
                    'center_shift': center - init_center,
                    'amplitude': amp,
                    'sigma': sigma,
                    'gamma': gamma if self.peak_shapes[i] == 'Voigt' else None,
                    'fwhm': fwhm,
                    'type': self.peak_types[i],
                    'shape': self.peak_shapes[i],
                    'tolerance': self.peak_tolerances[i]
                })
                idx += 3 if self.peak_shapes[i] != 'Voigt' else 4
                
            self.display_results()
            self.plot_data()
            self.calc_crystallinity_btn.setEnabled(True)
            self.save_btn.setEnabled(True)
            
        except Exception as e:
            QMessageBox.critical(self, "Error", f"Fitting failed:\n{str(e)}")
            
    def plot_fit_results(self, x_plot):
        if self.fit_results is None:
            return
            
        # Total fit
        y_fit = self.multi_peak_model(x_plot, *self.fit_results)
        self.ax.plot(x_plot, y_fit, 'r-', linewidth=2, label='Total fit', alpha=0.8)
        
        # Individual peaks
        n_peaks = len(self.initial_peak_centers)
        idx = 2
        
        for i in range(n_peaks):
            params_piece = np.zeros_like(self.fit_results)
            params_piece[0] = 0
            params_piece[1] = 0
            
            center = self.fit_results[idx]
            amp = self.fit_results[idx + 1]
            sigma = self.fit_results[idx + 2]
            
            params_piece[idx] = center
            params_piece[idx + 1] = amp
            params_piece[idx + 2] = sigma
            
            if self.peak_shapes[i] == 'Voigt' and len(self.fit_results) > idx + 3:
                params_piece[idx + 3] = self.fit_results[idx + 3]
                
            y_peak = self.multi_peak_model(x_plot, *params_piece)
            
            # Different styles
            linestyle = '-' if self.peak_types[i] == 'crystalline' else '--'
            alpha = 0.6 if self.peak_types[i] == 'crystalline' else 0.4
            
            self.ax.plot(x_plot, y_peak, color=self.colors[i % len(self.colors)], 
                        linestyle=linestyle, linewidth=1.5, alpha=alpha)
            
            idx += 3 if self.peak_shapes[i] != 'Voigt' else 4
            
        # Background
        background = self.fit_results[0] + self.fit_results[1] * x_plot
        self.ax.plot(x_plot, background, 'k:', linewidth=1, alpha=0.5, label='Background')
            
    def calculate_crystallinity(self):
        if self.fit_results is None or len(self.peak_params) == 0:
            return
            
        if self.fit_bounds[0] is not None and self.fit_bounds[1] is not None:
            mask = (self.x_data >= self.fit_bounds[0]) & (self.x_data <= self.fit_bounds[1])
            x_area = self.x_data[mask]
        else:
            x_area = self.x_data
            
        x_dense = np.linspace(x_area[0], x_area[-1], 10000)
        y_total = self.multi_peak_model(x_dense, *self.fit_results)
        
        y_crystalline = np.zeros_like(x_dense)
        idx = 2
        for i, params in enumerate(self.peak_params):
            if params['type'] == 'crystalline':
                center = self.fit_results[idx]
                amp = self.fit_results[idx + 1]
                sigma = self.fit_results[idx + 2]
                
                shape = self.peak_shapes[i]
                
                if shape == 'Gaussian':
                    y_crystalline += self.gaussian(x_dense, amp, center, sigma)
                elif shape == 'Lorentzian':
                    y_crystalline += self.lorentzian(x_dense, amp, center, sigma)
                else:
                    gamma = self.fit_results[idx + 3] if shape == 'Voigt' else sigma
                    y_crystalline += self.voigt(x_dense, amp, center, sigma, gamma)
                    
            idx += 3 if self.peak_shapes[i] != 'Voigt' else 4
            
        area_total = simpson(y_total, x_dense)
        area_crystalline = simpson(y_crystalline, x_dense)
        
        if area_total > 0:
            crystallinity = (area_crystalline / area_total) * 100
            self.crystallinity_result.setText(f"Crystallinity: {crystallinity:.2f}%")
        else:
            self.crystallinity_result.setText("Error: zero area")
            
    def display_results(self):
        """Display detailed peak information"""
        if not self.peak_params:
            return
            
        text = "="*70 + "\n"
        text += "FITTING RESULTS\n"
        text += "="*70 + "\n\n"
        
        # Background
        text += f"BACKGROUND: y = {self.fit_results[1]:.4f}·x + {self.fit_results[0]:.4f}\n\n"
        
        # Peak details
        text += "-"*70 + "\n"
        text += f"{'Peak':<5} {'Type':<12} {'Shape':<10} {'Position':<12} {'Shift':<10} {'Amplitude':<12} {'FWHM':<10}\n"
        text += "-"*70 + "\n"
        
        for i, p in enumerate(self.peak_params):
            type_short = p['type'][:3]
            text += f"{i+1:<5} {type_short:<12} {p['shape']:<10} "
            text += f"{p['center']:<12.4f} {p['center_shift']:<+10.4f} "
            text += f"{p['amplitude']:<12.4f} {p['fwhm']:<10.4f}\n"
        
        text += "-"*70 + "\n\n"
        
        # Detailed parameters per peak
        for i, p in enumerate(self.peak_params):
            text += f"Peak {i+1} [{p['type'].upper()}, {p['shape']}]:\n"
            text += f"  Initial center: {p['initial_center']:.4f}\n"
            text += f"  Final center: {p['center']:.4f} (shift: {p['center_shift']:+.4f})\n"
            text += f"  Tolerance: {p['tolerance']*100:.1f}%\n"
            text += f"  Amplitude: {p['amplitude']:.4f}\n"
            text += f"  Sigma (σ): {p['sigma']:.4f}\n"
            if p['gamma'] is not None:
                text += f"  Gamma (γ): {p['gamma']:.4f}\n"
            text += f"  FWHM: {p['fwhm']:.4f}\n\n"
        
        # Fit quality
        y_current = self.get_current_y_data()
        if self.fit_bounds[0] is not None and self.fit_bounds[1] is not None:
            mask = (self.x_data >= self.fit_bounds[0]) & (self.x_data <= self.fit_bounds[1])
            x_fit = self.x_data[mask]
            y_fit = y_current[mask]
        else:
            x_fit = self.x_data
            y_fit = y_current
            
        y_model = self.multi_peak_model(x_fit, *self.fit_results)
        residuals = y_fit - y_model
        rss = np.sum(residuals**2)
        tss = np.sum((y_fit - np.mean(y_fit))**2)
        r2 = 1 - (rss / tss) if tss != 0 else 0
        
        text += f"FIT QUALITY:\n"
        text += f"  R² = {r2:.6f}\n"
        text += f"  RMSE = {np.sqrt(np.mean(residuals**2)):.6f}\n"
        
        self.results_text.setText(text)
        
    def save_results(self):
        if self.fit_results is None:
            return
            
        filename, _ = QFileDialog.getSaveFileName(
            self, "Save results", "", "Text files (*.txt);;CSV files (*.csv)")
            
        if filename:
            try:
                with open(filename, 'w') as f:
                    f.write("# WAXS Multi-Peak Fitting Results\n")
                    f.write(f"# Date: {QDateTime.currentDateTime().toString()}\n")
                    f.write(f"# Peak shapes: Individual per peak\n")
                    f.write(f"# Number of peaks: {len(self.peak_params)}\n")
                    if self.fit_bounds[0] is not None:
                        f.write(f"# Fitting boundaries: [{self.fit_bounds[0]:.4f}, {self.fit_bounds[1]:.4f}]\n")
                    if self.background_subtracted:
                        f.write("# Background subtraction: Applied\n")
                    if self.lorentz_corrected:
                        f.write("# Lorentz correction: Applied\n")
                    f.write("\n")
                    
                    f.write("PEAK PARAMETERS:\n")
                    f.write("-"*100 + "\n")
                    header = "Peak\tType\tShape\tInit_pos\tFinal_pos\tShift\tTol(%)\tAmplitude\tSigma\tFWHM"
                    f.write(header + "\n")
                    
                    for i, p in enumerate(self.peak_params):
                        line = f"{i+1}\t{p['type']}\t{p['shape']}\t"
                        line += f"{p['initial_center']:.6f}\t{p['center']:.6f}\t"
                        line += f"{p['center_shift']:+.6f}\t{p['tolerance']*100:.1f}\t"
                        line += f"{p['amplitude']:.6f}\t{p['sigma']:.6f}\t{p['fwhm']:.6f}"
                        f.write(line + "\n")
                        
                    f.write("\nBACKGROUND:\n")
                    f.write("-"*100 + "\n")
                    f.write(f"Offset: {self.fit_results[0]:.6f}\n")
                    f.write(f"Slope: {self.fit_results[1]:.6f}\n")
                    
                    # Add crystallinity
                    f.write(f"\nCRYSTALLINITY: {self.crystallinity_result.text()}\n")
                    
                QMessageBox.information(self, "Success", f"Results saved to {filename}")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Could not save:\n{str(e)}")

def main():
    app = QApplication(sys.argv)
    window = PeakFittingApp()
    window.show()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()

SystemExit: 0

C:\Users\LabPCA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
